In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
    save_features_as_parquet,
)
from image_analysis_3D.featurization_utils.granularity_utils import (
    measure_3D_granularity,
)

# from granularity import measure_3D_granularity
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    start_profiling,
    stop_profiling,
)

image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "G9-7"
    patient = "NF0055_T1"
    channel = "AGP"
    compartment = "Cell"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{root_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)
channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)

In [3]:
# read in channel mapping
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

In [4]:
start_time, start_mem = start_profiling()

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
    mask_key_name=[channel_n_compartment_mapping[compartment]],
    raw_image_key_name=[channel_n_compartment_mapping[channel]],
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
if in_notebook:
    verbose = True
else:
    verbose = False
if processor_type == "CPU":
    object_measurements = measure_3D_granularity(
        object_loader=object_loader,
        radius=10,  # radius of the structuring element for background removal (CellProfiler default)
        granular_spectrum_length=16,  # range of the granular spectrum
        subsample_size=0.25,  # subsample the image for faster processing
        image_sample_size=0.25,  # further subsample for background removal
        mask_threshold=0.9,  # threshold for determining mask after interpolation
        verbose=verbose,
    )
else:
    raise ValueError(
        f"Processor type {processor_type} is not supported. Use 'CPU' only."
    )
final_df = pd.DataFrame(object_measurements)
# get the mean of each value in the array
# melt the dataframe to wide format
final_df = final_df.pivot_table(
    index=["object_id"], columns=["feature"], values=["value"]
)
final_df.columns = final_df.columns.droplevel()
final_df = final_df.reset_index()
# prepend compartment and channel to column names
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Granularity",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)
final_df.insert(0, "image_set", image_set_loader.image_set_name)
save_path = save_features_as_parquet(
    parent_path=output_parent_path,
    df=final_df,
    feature_type="Granularity",
    channel=channel,
    compartment=compartment,
    cpu_or_gpu=processor_type,
)
final_df.head()

Subsampled image: (121, 1527, 1528) -> (31, 382, 382) (factor=0.25)
Background subsampled: pixels (31, 382, 382) -> back_pixels (31, 382, 382) (image_sample_size=0.25)
Background removed via tophat filter.
Image startmean: 1042.909332, Processing 8 objects, Spectrum length: 16


Scale 1 - gs: 3.2581, currentmean: 1008.930338


Total measurements: 128
Non-zero measurements: 32
Mean granularity: 11.13


feature,image_set,object_id,Cell_AGP_Granularity_1,Cell_AGP_Granularity_2,Cell_AGP_Granularity_3,Cell_AGP_Granularity_4,Cell_AGP_Granularity_5,Cell_AGP_Granularity_6,Cell_AGP_Granularity_7,Cell_AGP_Granularity_8,Cell_AGP_Granularity_9,Cell_AGP_Granularity_10,Cell_AGP_Granularity_11,Cell_AGP_Granularity_12,Cell_AGP_Granularity_13,Cell_AGP_Granularity_14,Cell_AGP_Granularity_15,Cell_AGP_Granularity_16
0,G9-7,1,3.269702,0.664252,8.912072,0.0,0.0,0.0,0.0,0.0,21.505744,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,G9-7,2,8.295801,11.795346,12.576844,0.0,0.0,0.0,0.0,0.0,16.760849,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,G9-7,3,9.903264,2.974071,9.844658,0.0,0.0,0.0,0.0,0.0,19.125099,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,G9-7,4,2.360744,4.079215,12.649152,0.0,0.0,0.0,0.0,0.0,20.105866,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,G9-7,5,4.383249,2.814712,11.979320,0.0,0.0,0.0,0.0,0.0,20.091555,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
stop_profiling(
    start_time=start_time,
    start_mem=start_mem,
    feature_type="Granularity",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU=processor_type,
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Granularity_{processor_type}.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0055_T1
        Well and FOV: G9-7
        Feature type: Granularity
        CPU/GPU: CPU
        Peak memory (tracemalloc): 19309.27 MB
        Current memory (tracemalloc): 1078.27 MB
        RSS at end: 1274.24 MB
        Time elapsed:
        --- 522.14 seconds ---
        --- 8.70 minutes ---
        --- 0.15 hours ---
    


True